In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 6.2 Special Topics: Introduction to Neural Networks
### *Optional Module — Not Assessed*
- Neural networks basics
- Still follows `instantiate -> fit -> predict`!
- **Requires scaling** (unlike tree models)
- For tabular data, rarely beats Random Forest/XGBoost

## Neural Network Basics
- **Input layer**: One neuron per feature
- **Hidden layers**: Transform inputs through weights + activation functions
- **Output layer**: Final prediction
- Each neuron: output = activation(sum(weights * inputs) + bias)

## Let's Build One
- scikit-learn's `MLPClassifier` (Multi-Layer Perceptron)
- Same API: instantiate -> fit -> predict
- **Must scale features** (StandardScaler)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

train_df = pd.read_csv('../data/training.csv')
test_df = pd.read_csv('../data/testing.csv')
train_df['DEPARTED'] = (train_df['SEM_3_STATUS'] != 'E').astype(int)
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

numeric_features = ['HS_GPA','HS_MATH_GPA','HS_ENGL_GPA','UNITS_ATTEMPTED_1','UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1','UNITS_COMPLETED_2','DFW_UNITS_1','DFW_UNITS_2','GPA_1','GPA_2',
    'DFW_RATE_1','DFW_RATE_2','GRADE_POINTS_1','GRADE_POINTS_2']
categorical_features = ['RACE_ETHNICITY','GENDER','FIRST_GEN_STATUS','COLLEGE']

train_enc = pd.get_dummies(train_df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
test_enc = pd.get_dummies(test_df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)
train_enc = train_enc.fillna(train_enc.median())
test_enc = test_enc.fillna(test_enc.median())

X_train, y_train = train_enc, train_df['DEPARTED']
X_test, y_test = test_enc, test_df['DEPARTED']

# IMPORTANT: Neural networks REQUIRE scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

nn = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16),
    activation='relu',
    solver='adam',
    alpha=0.001,
    batch_size=32,
    learning_rate='adaptive',
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)

nn.fit(X_train_scaled, y_train)
nn_prob = nn.predict_proba(X_test_scaled)[:, 1]

print(f"Neural Network ROC-AUC: {roc_auc_score(y_test, nn_prob):.4f}")
print(f"\nNote: Neural networks require scaled features!")

## Neural Networks vs. Tree-Based Models
| Aspect | Neural Networks | Tree-Based Models |
|:-------|:---------------|:-----------------|
| Preprocessing | Requires scaling | None needed |
| Interpretability | Black box | Moderate to high |
| Tuning effort | High (many params) | Moderate |
| Tabular data | Rarely outperforms | Strong default |
| Images/text | Excellent | Not suitable |

## Key Takeaways
- Neural networks follow the same scikit-learn pattern
- They **require scaling** (unlike tree models)
- For tabular student data, rarely outperform RF/XGBoost
- Worth understanding for completeness
- Best suited for non-tabular data (images, text, etc.)

**Module 6 Complete! Next: Module 7 — EDA in Unstructured Data**